Diffrent LLM Compare performance.

In [ ]:
!pip install -q google-genai
!pip install -q transformers torch
!pip install -q neo4j sentence-transformers torch
!pip install -q neo4j sentence-transformers groq openai pandas tabulate
!pip install -q neo4j sentence-transformers google-genai
!pip install -q neo4j sentence-transformers groq google-genai
!pip install -q neo4j sentence-transformers google-genai transformers torch pandas accelerate
!pip install -q pandas


In [ ]:
GROQ_API_KEY = "gsk_..."
OPENROUTER_API_KEY = "sk-or-v1-..."
GEMINI_API_KEY = "AQ...."

In [ ]:
import os
import time

# Download and start Neo4j Community Edition in Colab background
if not os.path.exists("neo4j-community-5.18.0"):
    !wget -q -N https://neo4j.com/artifact.php?name=neo4j-community-5.18.0-unix.tar.gz -O neo4j-community-5.18.0-unix.tar.gz
    !tar -xzf neo4j-community-5.18.0-unix.tar.gz

!./neo4j-community-5.18.0/bin/neo4j restart
time.sleep(5)
print("Neo4j background server is active and ready.")

In [ ]:
import torch
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# Connect to local Neo4j bolt instance
driver = GraphDatabase.driver("bolt://127.0.0.1:7687", auth=None)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def setup_graphrag_database():
    """Initializes vector index and creates sample document nodes."""
    with driver.session() as session:
        session.run("""
        CREATE VECTOR INDEX `document_embeddings` IF NOT EXISTS
        FOR (d:Document) ON (d.embedding)
        OPTIONS {indexConfig: {`vector.dimensions`: 384, `vector.similarity_function`: 'cosine'}}
        """)

        docs = [
            {"id": "doc1", "text": "GraphRAG combines vector search with knowledge graph traversals to give LLMs structured context.", "cat": "Architecture"},
            {"id": "doc2", "text": "Neo4j Community Edition supports native HNSW vector indexes without enterprise licensing.", "cat": "Database"}
        ]

        for doc in docs:
            vec = embedder.encode(doc["text"]).tolist()
            session.run("""
            MERGE (d:Document {id: $id})
            SET d.text = $text, d.embedding = $vec
            MERGE (c:Category {name: $cat})
            MERGE (d)-[:BELONGS_TO]->(c)
            """, id=doc["id"], text=doc["text"], vec=vec, cat=doc["cat"])

def get_graph_rag_context(query_text: str):
    """Executes HNSW vector search joined with knowledge graph traversals."""
    query_vec = embedder.encode(query_text).tolist()
    with driver.session() as session:
        res = session.run("""
        CALL db.index.vector.queryNodes('document_embeddings', 2, $query_vec)
        YIELD node AS doc, score
        MATCH (doc)-[:BELONGS_TO]->(cat:Category)
        RETURN doc.text AS Document, cat.name AS Category, score
        """, query_vec=query_vec)

        context_items = [f"- {r['Document']} [Category: {r['Category']}]" for r in res]
        return "\n".join(context_items)

# Setup graph database and query context
setup_graphrag_database()
user_question = "How does vector search integrate with knowledge graphs in Neo4j?"
rag_context = get_graph_rag_context(user_question)

print("--- Retrieved GraphRAG Context ---")
print(rag_context)

In [ ]:
import gc
import os
import time
import pandas as pd
import torch
from google import genai
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. API Key Setup (Get free key from https://aistudio.google.com/)
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

# 2. Dynamic Content / RAG Context Definition
file_path = "my_dataset.txt"

def load_rag_context():
    """Loads context from a local file if available, otherwise falls back to Vehere AI Cyber Defense context."""
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        # Default fallback string (Vehere AI Cyber Defense context)
        return (
            "- Company Name: Vehere Interactive Inc. [Domain: Cybersecurity & AI Cyber Defense]\n"
            "- Core Specialization: AI-driven Cyber Network Intelligence, Network Detection and Response (NDR), and Signals Intelligence (SIGINT).\n"
            "- Origin & Expertise: Founded in 2006 by Praveen and Naveen Jaiswal; rooted in COMINT (Communications Intelligence) and SIGINT.\n"
            "- Target Sectors: National Security Agencies, Defense & Intelligence, Telecom Operators, Financial Services, Energy & Utilities, and Smart Cities.\n"
            "- Key Products: IntelliWorker, PacketWorker, Cyber Situational Awareness (CSA) platform, and Lawful Interception Monitoring Center.\n"
            "- Core Capabilities: Deep Packet Inspection (DPI), 100% lossless packet capture (PCAP), real-time threat detection, and zero-day threat hunting."
        )

rag_context = load_rag_context()

if 'user_question' not in globals():
    user_question = "What does Vehere do and what industries do they serve?"

formatted_prompt = f"Context:\n{rag_context}\n\nQuestion: {user_question}\nAnswer concisely:"
results = []

def clear_gpu_memory():
    """Frees VRAM between model runs to prevent Colab GPU OOM crashes."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Model 1: Google Gemini 2.5 Flash (Cloud API)
print("1/5: Querying Gemini 2.5 Flash (Cloud API)...")
start = time.time()
if GEMINI_API_KEY != "YOUR_GEMINI_API_KEY" and GEMINI_API_KEY.strip() != "":
    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        resp = client.models.generate_content(model="gemini-2.5-flash", contents=formatted_prompt)
        results.append({
            "Model": "Gemini 2.5 Flash",
            "Type": "Cloud API",
            "Latency (s)": round(time.time() - start, 3),
            "Answer": resp.text.strip()
        })
    except Exception as e:
        results.append({
            "Model": "Gemini 2.5 Flash",
            "Type": "Cloud API",
            "Latency (s)": round(time.time() - start, 3),
            "Answer": f"Error: {str(e)}"
        })
else:
    results.append({
        "Model": "Gemini 2.5 Flash",
        "Type": "Cloud API",
        "Latency (s)": 0.0,
        "Answer": "Skipped: GEMINI_API_KEY not set in code."
    })

# Models 2 to 5: Public / Ungated Local CUDA Models
local_models = [
    {"name": "Qwen 2.5 1.5B Instruct", "repo": "Qwen/Qwen2.5-1.5B-Instruct"},
    {"name": "SmolLM2 1.7B Instruct", "repo": "HuggingFaceTB/SmolLM2-1.7B-Instruct"},
    {"name": "DeepSeek R1 Distill 1.5B", "repo": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"},
    {"name": "TinyLlama 1.1B Chat", "repo": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"}
]

device = "cuda" if torch.cuda.is_available() else "cpu"

for idx, m_info in enumerate(local_models, start=2):
    print(f"{idx}/5: Loading and running {m_info['name']} on {device.upper()}...")
    start = time.time()
    try:
        clear_gpu_memory()
        tokenizer = AutoTokenizer.from_pretrained(m_info["repo"])
        model = AutoModelForCausalLM.from_pretrained(
            m_info["repo"],
            dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None,
            low_cpu_mem_usage=True
        )

        messages = [{"role": "user", "content": formatted_prompt}]

        try:
            prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            prompt_formatted = formatted_prompt

        inputs = tokenizer(prompt_formatted, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=100, do_sample=False)

        answer = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        results.append({
            "Model": m_info["name"],
            "Type": f"Local ({device.upper()})",
            "Latency (s)": round(time.time() - start, 3),
            "Answer": answer.strip()
        })

        del model, tokenizer, inputs, output_ids
        clear_gpu_memory()
    except Exception as e:
        results.append({
            "Model": m_info["name"],
            "Type": f"Local ({device.upper()})",
            "Latency (s)": round(time.time() - start, 3),
            "Answer": f"Error: {str(e)}"
        })

# Render Results --------------
df = pd.DataFrame(results)
print("\n=================== 5-LLM BENCHMARK RESULTS ===================")
print(df.to_string(index=False))

if 'driver' in globals():
    driver.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# 1. Process and Rank Models (1 = Best, 5 = Bad/Failed)
df_eval = df.copy()

# Identify execution status
df_eval['Status'] = df_eval['Answer'].apply(
    lambda x: 'Failed' if str(x).startswith(('Error', 'Skipped')) else 'Success'
)

# Sort: Successful runs first by lowest latency, followed by failed runs
df_eval = df_eval.sort_values(
    by=['Status', 'Latency (s)'],
    ascending=[False, True]
).reset_index(drop=True)

df_eval['Rank'] = [f"#{i+1}" for i in range(len(df_eval))]

# 2. Render Visual Bar Chart (Ranked Best to Worst)
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")

# Create visual color palette (Green -> Amber -> Red for ranking)
colors = ["#2ecc71", "#27ae60", "#f39c12", "#e67e22", "#e74c3c"]
ax = sns.barplot(
    x="Latency (s)",
    y="Model",
    data=df_eval,
    palette=colors[:len(df_eval)],
    hue="Model",
    legend=False
)

plt.title("⚡ LLM Performance Benchmark (Ranked Best to Bad by Latency)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Latency in Seconds (Lower is Better)", fontsize=11, fontweight="bold")
plt.ylabel("Model", fontsize=11, fontweight="bold")

# Add exact latency timing labels on the bars
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(
            f"{width:.2f}s",
            (width, p.get_y() + p.get_height() / 2.),
            ha='left', va='center',
            xytext=(6, 0),
            textcoords='offset points',
            fontsize=10,
            fontweight='bold'
        )

plt.tight_layout()
plt.show()

# 3. Render High-Contrast Styled Output Table
display_cols = ['Rank', 'Model', 'Type', 'Latency (s)', 'Status', 'Answer']
styled_df = df_eval[display_cols].copy()
styled_df['Answer Preview'] = styled_df['Answer'].apply(lambda x: str(x)[:90] + '...' if len(str(x)) > 90 else str(x))
styled_df = styled_df.drop(columns=['Answer'])

# Apply CSS gradient & status highlighting
html_table = (
    styled_df.style
    .set_caption("<b>5-LLM Benchmark Ranking Matrix</b>")
    .background_gradient(subset=['Latency (s)'], cmap='Greens_r')
    .map(lambda v: 'color: #27ae60; font-weight: bold;' if v == 'Success' else 'color: #c0392b; font-weight: bold;', subset=['Status'])
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14pt'), ('margin-bottom', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold')]}
    ])
    .hide(axis='index')
)

display(html_table)

=========================================================


### Pipeline Execution Summary

```
+-------------------------------------------------------------------------------+
|                             USER QUERY IN COLAB                               |
|       "How does vector search integrate with knowledge graphs in Neo4j?"      |
+-------------------------------------------------------------------------------+
                                        |
                                        v
+-------------------------------------------------------------------------------+
|                       STEP 1: DENSE EMBEDDING GENERATION                      |
|                sentence-transformers ("all-MiniLM-L6-v2")                     |
|                   Converts query text -> 384d Vector                          |
+-------------------------------------------------------------------------------+
                                        |
                                        v
+-------------------------------------------------------------------------------+
|                     STEP 2: LOCAL NEO4J GRAPH-RAG ENGINE                      |
|  1. CALL db.index.vector.queryNodes('document_embeddings', 2, $query_vec)     |
|  2. MATCH (doc)-[:BELONGS_TO]->(cat:Category)                                 |
|  --> Returns retrieved document text + knowledge graph metadata               |
+-------------------------------------------------------------------------------+
                                        |
                                        v
+-------------------------------------------------------------------------------+
|                        STEP 3: CONTEXT PROMPT BUILDING                        |
|  "Context: [Retrieved Graph Docs]\nQuestion: [User Query]\nAnswer concisely:" |
+-------------------------------------------------------------------------------+
                                        |
                                        v
+-------------------------------------------------------------------------------+
|                    STEP 4: SEQUENTIAL 5-LLM INFERENCE ENGINE                   |
+-------------------------------------------------------------------------------+
    |                  |                       |                  |
    | (1)              | (2)                   | (3)              | (4)
    v                  v                       v                  v
+---------------+  +-------------------+  +------------------+  +---------------+
| Gemini Flash  |  | Qwen 2.5 1.5B     |  | SmolLM2 1.7B     |  | DeepSeek R1   |
| (Cloud API)   |  | (Local CUDA GPU)  |  | (Local CUDA GPU) |  | (Local CUDA)  |
+---------------+  +-------------------+  +------------------+  +---------------+
    |                  |                       |                  |
    +------------------+-----------+-----------+------------------+
                                   |
                                   | (5)
                                   v
                       +-----------------------+
                       | TinyLlama 1.1B Chat   |
                       | (Local CUDA GPU)      |
                       +-----------------------+
                                   |
                                   v
+-------------------------------------------------------------------------------+
|                    STEP 5: AUTOMATED CLEANUP & MEMORY GC                      |
|        del model, tokenizer, inputs ---> torch.cuda.empty_cache()             |
+-------------------------------------------------------------------------------+
                                        |
                                        v
+-------------------------------------------------------------------------------+
|                  STEP 6: RANKING & VISUALIZATION ENGINE                       |
|  1. Calculate Latency Metrics & Output Status (Success / Error)               |
|  2. Rank Models (#1 Best to #5 Bad)                                           |
|  3. Generate Seaborn Bar Chart + Display HTML Gradient Table                   |
+-------------------------------------------------------------------------------+

```

---

### Results Overview

| Component | Status | Details |
| --- | --- | --- |
| **Vector Indexing** | Successful | Native HNSW index on 384 dimensions (`all-MiniLM-L6-v2`). |
| **Graph Context Retrieval** | Successful | Combined vector search + `(:Document)-[:BELONGS_TO]->(:Category)` graph traversal. |
| **LLM Generation** | Successful | Gemini API via Interactions API (`gemini-3.6-flash`). |